# TTC Subway Delay Data — Preprocessing & Exploratory Analysis

**Author:** Umniyah Aalam
**Date:** February 2026  
**Dataset:** Toronto Transit Commission (TTC) subway delay records, 2024–2025  

## Objective

Prepare TTC subway delay data for a regression model that predicts **delay duration in minutes** for a given line, station, time, and day.

## Approach

| Step | Description |
|------|-------------|
| **1. Data Loading** | Combine 2024 (Excel) and 2025 (CSV) delay records into a single dataset |
| **2. Data Cleaning** | Standardize line names, remove invalid records, handle outliers, drop irrelevant columns |
| **3. Feature Engineering** | Extract temporal features, compute historical average delays per route/time |
| **4. Correlation Analysis** | Identify data leakage, multicollinearity, and useful predictors |
| **5. Exploratory Data Analysis** | Visualize delay patterns by time, day, line, station, and incident code |
| **6. Export** | Save the cleaned dataset for model training |

## Target Variable

`min_delay_capped` — delay in minutes, capped at 60 to mitigate extreme outliers.

---

### 1. Data Loading

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

In [16]:
df_2024 = pd.read_excel("./data/ttc-subway-delay-2024.xlsx", sheet_name="Subway")
df_2025 = pd.read_csv("./data/ttc-subway-delay-data-since-2025.csv")

In [17]:
# pip install openpyxl

In [18]:
df = pd.concat([df_2024, df_2025], ignore_index = True)
print(f"Combind dataset: {df.shape[0]:,} records, {df.shape[1]} columns")

Combind dataset: 52,180 records, 11 columns


In [19]:
df.head()

,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle,_id
0,2024-01-01 00:00:00,02:00,Monday,SHEPPARD STATION,MUI,0,0,N,YU,5491,NaN
1,2024-01-01 00:00:00,02:00,Monday,DUNDAS STATION,MUIS,0,0,N,YU,0,NaN
2,2024-01-01 00:00:00,02:08,Monday,DUNDAS STATION,MUPAA,4,10,N,YU,6051,NaN
3,2024-01-01 00:00:00,02:13,Monday,KENNEDY BD STATION,PUTDN,10,16,E,BD,5284,NaN
4,2024-01-01 00:00:00,02:22,Monday,BLOOR STATION,MUPAA,4,10,N,YU,5986,NaN


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52180 entries, 0 to 52179
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       52180 non-null  object 
 1   Time       52180 non-null  object 
 2   Day        52180 non-null  object 
 3   Station    52180 non-null  object 
 4   Code       52180 non-null  object 
 5   Min Delay  52180 non-null  int64  
 6   Min Gap    52180 non-null  int64  
 7   Bound      33198 non-null  object 
 8   Line       52069 non-null  object 
 9   Vehicle    52180 non-null  int64  
 10  _id        25713 non-null  float64
dtypes: float64(1), int64(3), object(7)
memory usage: 4.4+ MB


### 2. Data Cleaing

In [21]:
print(f"Unique Line values before cleaning: {df['Line'].unique()}")
df['Line'].value_counts()

Unique Line values before cleaning: ['YU' 'BD' 'YUS' 'YU/BD' 'SHP' nan 'BLOOR DANFORTH' 'YU / BD' 'YU/ BD'
 'SRT' 'YUS/BD' 'SHEP' 'LINE 1' 'TRACK LEVEL ACTIVITY' 'YU & BD'
 '109 RANEE' 'ONGE-UNIVERSITY AND BL' 'YU/BD/SHP' 'BD/ YUS' 'BD/ YU'
 'BD/YU' 'BD / YU' '20 CLIFFSIDE' 'YUS/ BD' 'BD/YUS' 'YUS/ BD/ SHP'
 'YUS / BD' 'YU -BD LINES' '29 DUFFERIN' 'YU/BD LINES']


Line
YU                        27221
BD                        22133
SHP                        1955
YU/BD                       655
YUS/BD                       24
YU/ BD                       20
YUS                          11
YU / BD                      11
BD/YU                         7
YU & BD                       6
BD/YUS                        3
SRT                           3
YU/BD/SHP                     2
YUS / BD                      2
YU/BD LINES                   2
SHEP                          1
BLOOR DANFORTH                1
TRACK LEVEL ACTIVITY          1
LINE 1                        1
ONGE-UNIVERSITY AND BL        1
109 RANEE                     1
BD / YU                       1
BD/ YU                        1
BD/ YUS                       1
20 CLIFFSIDE                  1
YUS/ BD/ SHP                  1
YUS/ BD                       1
YU -BD LINES                  1
29 DUFFERIN                   1
Name: count, dtype: int64

In [22]:
line_mapping = {
    # Line 1 — Yonge-University
    'YU': 'Line 1', 'YUS': 'Line 1', 'LINE 1': 'Line 1',
    # Line 2 — Bloor-Danforth
    'BD': 'Line 2', 'BLOOR DANFORTH': 'Line 2',
    # Line 4 — Sheppard
    'SHP': 'Line 4', 'SHEP': 'Line 4',
    # Line 3 — Scarborough RT (decommissioned 2023)
    'SRT': 'Line 3',
    # Multi-line: Line 1 / Line 2
    'YU/BD': 'Line 1/2', 'YU/ BD': 'Line 1/2', 'YU / BD': 'Line 1/2',
    'YU & BD': 'Line 1/2', 'YU/BD LINES': 'Line 1/2', 'YU -BD LINES': 'Line 1/2',
    'BD/YU': 'Line 1/2', 'BD/ YU': 'Line 1/2', 'BD / YU': 'Line 1/2',
    'YUS/BD': 'Line 1/2', 'YUS / BD': 'Line 1/2', 'YUS/ BD': 'Line 1/2',
    'BD/YUS': 'Line 1/2', 'BD/ YUS': 'Line 1/2', 'ONGE-UNIVERSITY AND BL': 'Line 1/2',
    # Multi-line: Line 1 / Line 2 / Line 4
    'YU/BD/SHP': 'Line 1/2/4', 'YUS/ BD/ SHP': 'Line 1/2/4',
    # Invalid — bus routes and non-subway entries
    '109 RANEE': None, '20 CLIFFSIDE': None, '29 DUFFERIN': None, 'TRACK LEVEL ACTIVITY': None,
}

In [23]:
df['Line'] = df['Line'].map(line_mapping)
print(f"Unique Line values after mapping: {df['Line'].dropna().nunique()}")
df['Line'].value_counts()

Unique Line values after mapping: 6


Line
Line 1        27233
Line 2        22134
Line 4         1956
Line 1/2        736
Line 3            3
Line 1/2/4        3
Name: count, dtype: int64

In [24]:
df.head()

,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle,_id
0,2024-01-01 00:00:00,02:00,Monday,SHEPPARD STATION,MUI,0,0,N,Line 1,5491,NaN
1,2024-01-01 00:00:00,02:00,Monday,DUNDAS STATION,MUIS,0,0,N,Line 1,0,NaN
2,2024-01-01 00:00:00,02:08,Monday,DUNDAS STATION,MUPAA,4,10,N,Line 1,6051,NaN
3,2024-01-01 00:00:00,02:13,Monday,KENNEDY BD STATION,PUTDN,10,16,E,Line 2,5284,NaN
4,2024-01-01 00:00:00,02:22,Monday,BLOOR STATION,MUPAA,4,10,N,Line 1,5986,NaN
